# Ishu's DataCheck Bot
Built by Iswarya Subburaj (Ishu), with AI assistance.

Ask questions about a table. **Python checks, counts and calculates. It writes every answer.**
For harder questions the AI makes a plan, and Python runs it. The AI receives your question, column names and column types, not your rows. Read the "I understood" line to check each AI answer.

Click **Run all**, then scroll down to the chat window.

## 1. Practice data (errors added on purpose)

In [ ]:
import re
import pandas as pd

data = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 6, 8, 8],
    "city":        ["Berlin", "Munich", "berlin ", None, "Hamburg", "Munich", "Munich", "Berlin", "Hamburg"],
    "age":         [25, 31, 42, 29, -5, 38, 38, 33, 27],
    "balance":     [1200, 1500, 1100, 1300, 1250, 1400, 1400, 99000, 1350],
    "join_date":   ["2025-01-10", "2025-02-11", "2025-03-12", "2025-13-40", "2025-05-14",
                    "2025-06-15", "2025-06-15", "2025-07-16", "2025-08-17"],
})
data

## 2. The checks

In [ ]:
def rows(mask):
    return [int(i) for i in mask[mask].index]

def numbers(df, col):
    return pd.to_numeric(df[col], errors="coerce")

def missing(df, col):
    return rows(df[col].isna())

def duplicate_rows(df, col=None):
    return rows(df.duplicated(keep=False))

def duplicate_ids(df, col):
    return rows(df[col].notna() & df[col].duplicated(keep=False))

def negative(df, col):
    return rows(numbers(df, col) < 0)

def unusual(df, col):
    n = numbers(df, col)
    low, high = n.quantile(0.25), n.quantile(0.75)
    gap = 1.5 * (high - low)
    return rows((n < low - gap) | (n > high + gap))

def spelling(df, col):
    text = df[col].dropna().astype(str)
    return rows(text.groupby(text.str.strip().str.lower()).transform("nunique") > 1)

def bad_dates(df, col):
    parsed = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")
    return rows(df[col].notna() & parsed.isna())

def too_young(df, col):
    return rows(numbers(df, col) < 18)

def too_old(df, col):
    return rows(numbers(df, col) > 100)

# check name: (function, words used in answers)
CHECKS = {
    "missing": (missing, "missing values"),
    "duplicate_rows": (duplicate_rows, "rows that appear more than once"),
    "duplicate_ids": (duplicate_ids, "rows with a repeated ID"),
    "negative": (negative, "negative values"),
    "unusual": (unusual, "unusual values (may be correct)"),
    "spelling": (spelling, "rows with mixed spelling or spacing"),
    "bad_dates": (bad_dates, "invalid dates (expected YYYY-MM-DD)"),
    "too_young": (too_young, "values under 18 (may be correct)"),
    "too_old": (too_old, "values over 100 (may be correct)"),
}

def columns_for(df, check):
    """Which columns does a check make sense for?"""
    is_id = lambda c: re.search(r"(^|_)id$|Id$|^ID$", str(c)) is not None
    number = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and not is_id(c)]
    text = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    date = [c for c in df.columns if "date" in str(c).lower()]
    age = [c for c in df.columns if str(c).lower() == "age"]
    return {
        "missing": list(df.columns), "duplicate_ids": [c for c in df.columns if is_id(c)],
        "negative": number, "unusual": number, "spelling": [c for c in text if c not in date],
        "bad_dates": date, "too_young": age, "too_old": age,
    }[check]

## 3. Understand the question

In [ ]:
WORDS = [   # order matters: the first match wins
    ("duplicate_ids", r"(duplicate|repeated)\w* ids?\b"),
    ("duplicate_rows", r"duplicate|repeated"),
    ("missing", r"missing|empty|blank|null"),
    ("negative", r"negative|below zero|below 0"),
    ("unusual", r"unusual|outlier|strange"),
    ("spelling", r"spell"),
    ("bad_dates", r"\bdates?\b"),
    ("too_young", r"too young|under 18|below 18"),
    ("too_old", r"too old|over 100|above 100"),
]

def find_column(question, df):
    for c in df.columns:
        for name in {str(c).lower(), str(c).lower().replace("_", " ")}:
            plural = name[:-1] + "ies" if name.endswith("y") else name + "s"
            if re.search(rf"\b({re.escape(name)}|{re.escape(plural)})\b", question):
                return c
    return None

def simple_understand(question, df):
    q = question.lower().strip()
    if q in ("help", "?"):
        return {"action": "help"}
    if re.search(r"check (all|everything|my data|the data)|^check$", q):
        return {"action": "check_all"}
    if re.search(r"summary|columns|how many rows", q) and not re.search(r"missing|duplicate", q):
        return {"action": "summary"}
    row = re.search(r"\brow (\d+)", q)
    if row:
        return {"action": "show_row", "row": int(row[1])}
    for check, pattern in WORDS:
        if re.search(pattern, q):
            return {"action": "check", "check": check, "column": find_column(q, df)}
    return None

## Simple questions (counts and averages)

In [ ]:
# Simple questions: counts, averages, highest, lowest, totals. Everything is calculated here, on your table.
COUNT_WORDS = r"\b(how many|count|number of|most common)\b"
STATS = {"average": "mean", "mean": "mean", "median": "median", "highest": "max", "maximum": "max",
         "max": "max", "lowest": "min", "minimum": "min", "min": "min", "total": "sum", "sum": "sum"}
STAT_NAMES = {"mean": "Average", "median": "Median", "max": "Highest", "min": "Lowest", "sum": "Total"}
CHECK_WORDS = r"missing|empty|blank|null|duplicate|repeated|negative|unusual|outlier|strange|spell|invalid|\bcheck\b"
COMPARE = {"above": ">", "over": ">", "more than": ">", "greater than": ">", "older than": ">",
           "under": "<", "below": "<", "less than": "<", "younger than": "<",
           "equal to": "==", "is": "==", "=": "==", "": "=="}
FILLER = set("""a an the are is was were be there of in on for with who whose which what where do does did
have has had and we i me my our your you it they them their this that these those all show give tell find please
students student people person persons customers customer rows row records record entries entry values value
column data dataset table file total from live lives""".split())

def clean(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()

def pattern_for(words):
    plural = words[:-1] + "ies" if words.endswith("y") else words + "s"
    return rf"(?<!\w)(?:{re.escape(words)}|{re.escape(plural)})(?!\w)"

def data_question(question, df):
    """Return an answer for a counting or number question, or None if it is another kind of question."""
    q = clean(re.sub(r"[?!,]|\.(?!\d)|'s\b", " ", question.lower()))
    counting = re.search(COUNT_WORDS, q) is not None
    stat_words = [w for w in STATS if re.search(rf"\b{w}\b", q)]
    if not counting and not stat_words:
        return None
    if re.search(CHECK_WORDS, q):
        text = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        found = {clean(v) for c in text if df[c].nunique() <= 200 for v in df[c].dropna().unique()}
        if counting and any(v and v not in FILLER and re.search(pattern_for(v), q) for v in found):
            return "I cannot mix a data check with a filter yet. Ask them one at a time."
        return None
    stats = [] if counting else sorted({STATS[w] for w in stat_words})
    q = clean(re.sub(COUNT_WORDS, " ", q))
    for word in stat_words:
        q = re.sub(rf"\b{word}\b", " ", q)

    names = {c: sorted({clean(c), clean(str(c).replace("_", " "))}, key=len, reverse=True) for c in df.columns}

    # 1. Number conditions, such as "age above 25"
    conditions = []
    operators = "|".join(re.escape(o) for o in sorted(COMPARE, key=len, reverse=True) if o)
    for col in df.columns:
        for name in names[col]:
            found = re.search(rf"(?<!\w){re.escape(name)}s?\s*(?:is\s+)?({operators})?\s*(-?\d+(?:\.\d+)?)(?!\w)", q)
            if found:
                conditions.append((col, COMPARE[found[1] or ""], float(found[2])))
                q = q[:found.start()] + " " + q[found.end():]
                break
    age = next((c for c in df.columns if clean(c) == "age"), None)
    found = re.search(r"\b(older than|younger than)\s*(\d+)\b|\b(under|below) (18)\b", q)
    if found and age is not None:
        conditions.append((age, COMPARE[found[1] or found[3]], float(found[2] or found[4])))
        q = q[:found.start()] + " " + q[found.end():]

    # 2. Column names and values written in the question (longest first)
    options = [(len(n), 0, n, c, None) for c in df.columns for n in names[c]]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col]):
            continue
        values = {clean(v) for v in df[col].dropna().unique()}
        if len(values) <= 200:
            options += [(len(v), 1, v, col, v) for v in values if v]
    options.sort(key=lambda o: (-o[0], o[1]))
    columns, hits = [], {}
    for _, _, words, col, value in options:
        if value in FILLER:
            continue
        if re.search(pattern_for(words), q):
            if value is None:
                columns.append(col)
                q = re.sub(pattern_for(words), " ", q)
            else:
                hits.setdefault(words, []).append(col)
    values = {}
    for words, cols in hits.items():
        if not re.search(pattern_for(words), q):
            continue   # already used by a longer name or value
        named = [c for c in cols if c in columns]
        if len(named) == 1:
            cols = named
        if len(cols) > 1:
            return f"'{words}' appears in several columns: {', '.join(map(str, cols))}. Add one column name."
        values.setdefault(cols[0], []).append(words)
        q = re.sub(pattern_for(words), " ", q)

    # 3. Never answer if part of the question was not understood
    unknown = [w for w in q.split() if w not in FILLER]
    if unknown:
        return (f"I am not sure about: {', '.join(unknown)}. I did not answer, to avoid a wrong result.\n"
                "Use exact column names and values. Type summary to see the columns.")

    keep = pd.Series(True, index=df.index)
    used = []
    for col, sign, number in conditions:
        n = numbers(df, col)
        keep &= (n > number) if sign == ">" else (n < number) if sign == "<" else (n == number)
        used.append(f"{col} {'=' if sign == '==' else sign} {number:g}")
    several = {}
    for col, wanted in values.items():
        column = df[col].map(clean, na_action="ignore")
        keep &= column.isin(wanted)
        used.append(f"{col} = {' or '.join(wanted)}")
        if len(wanted) > 1:
            several[col] = wanted
    others = [c for c in columns if c not in values]
    where = f" where {' and '.join(used)}" if used else ""
    table = df[keep]

    if stats:
        if len(stats) > 1:
            return "Ask for one calculation at a time."
        if len(others) != 1:
            if stats == ["sum"] and not others and used:
                return f"Count{where}: {len(table)} of {len(df)} rows."
            return "Name one number column, for example: average age"
        col, kind = others[0], stats[0]
        n = numbers(table, col).dropna()
        if n.empty:
            return f"I found no numbers in '{col}'{where}."
        skipped = len(table) - len(n)
        note = f" {skipped} rows without a number were left out." if skipped else ""
        result = round(float(getattr(n, kind)()), 2)
        shown = f"{result:.0f}" if result.is_integer() else f"{result}"
        return f"{STAT_NAMES[kind]} {col}{where}: {shown} (from {len(n)} rows).{note}"

    if not used and not others:
        return None
    lines = []
    if used:
        share = 100 * len(table) / len(df) if len(df) else 0
        lines.append(f"Count{where}: {len(table)} of {len(df)} rows ({share:.1f}%).")
    for col in list(several) + others:
        counts = table[col].map(clean, na_action="ignore").fillna("(empty)").value_counts()
        lines.append(f"{col}:")
        lines += [f"- {value}: {total}" for value, total in counts.head(15).items()]
        if len(counts) > 15:
            lines.append(f"(showing the top 15 of {len(counts)} different values)")
    return "\n".join(lines)

## 4. Answer (Python writes every number)

In [ ]:
HELP = """Try:
  check all
  any missing city?
  any duplicate rows?
  check spelling in city
  show row 4
  how many female?
  how many age above 25?
  average age
  count city
With AI on, ask in your own words:
  which city has the highest average balance?
  what percentage of Berlin customers are under 30?
  show the 3 oldest customers
  summary
Use the CSV button to load your own public or fake data.
Row numbers start at 0."""

def describe(df, check, col):
    found = CHECKS[check][0](df, col)
    label = CHECKS[check][1]
    place = f"{col}: " if col is not None else ""
    if not found:
        return f"{place}no {label} found."
    shown = ", ".join(map(str, found[:10]))
    more = f" and {len(found) - 10} more" if len(found) > 10 else ""
    return f"{place}{label}: {len(found)}. Rows: {shown}{more}."

def check_all(df):
    lines = [describe(df, "duplicate_rows", None)]
    for check in CHECKS:
        if check == "duplicate_rows":
            continue
        for col in columns_for(df, check):
            if CHECKS[check][0](df, col):
                lines.append(describe(df, check, col))
    return "\n".join(lines)

## AI planning (Python calculates)

In [ ]:
# The AI turns a question into a plan. It gets the question, column names and column types, never your rows.
import json

MODELS = ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.1-flash-lite"]
CHECK_NAMES = ["missing", "duplicate_rows", "duplicate_ids", "negative", "unusual",
               "spelling", "bad_dates", "too_young", "too_old"]

def obj(**properties):
    return {"type": "object", "properties": properties, "required": list(properties)}

def one_of(*choices):
    return {"type": "string", "enum": list(choices)}

TEXT = {"type": "string"}
FILTERS = {"type": "array", "items": obj(
    column=TEXT, op=one_of("=", "!=", ">", ">=", "<", "<=", "contains", "empty", "not_empty"),
    values={"type": "array", "items": TEXT})}

PLAN_SCHEMA = obj(
    action=one_of("calculate", "check", "cannot_answer"),
    calculation=one_of("count", "percent", "sum", "mean", "median", "min", "max",
                       "distinct", "list", "correlation", "none"),
    check=one_of(*CHECK_NAMES, "none"),
    column=TEXT, second_column=TEXT, group_by=TEXT,
    filters=FILTERS, among=FILTERS, match=one_of("all", "any"),
    order=one_of("largest", "smallest"), limit={"type": "integer"}, reason=TEXT)

INSTRUCTIONS = """Turn the user's question about one table into a plan. Python will run it.
Rules:
- Use only the listed column names. Use "" for unused text fields, [] for unused lists, 0 for no limit.
- Never drop or change a condition. If the plan cannot express the whole question, use cannot_answer and say why.
- Use cannot_answer for why, prediction, advice, or anything that is not in the columns.
- filters pick rows. Several values in one filter mean any of them. match says if all or any filters must be true.
- percent: filters is the part being measured, among is the group it is measured within (empty means all rows).
  Example: percentage of women who survived: filters Survived = 1, among Sex = female.
- group_by gives one result per group. "Which X has the most Y": calculation sum of Y, group_by X, order largest, limit 1.
- "Most common X": count, group_by X, order largest, limit 1.
- list shows rows, sorted by column if given. correlation uses column and second_column.
- check is only for data quality questions: set action check, the check name and optionally column.
- You cannot see the values. Write text values as the user said them; for yes/no meaning in a number column use 1 or 0."""

def ai_plan(question, df, client):
    columns = [{"name": str(c), "type": "number" if pd.api.types.is_numeric_dtype(df[c]) else "text"}
               for c in df.columns]
    prompt = f"{INSTRUCTIONS}\nColumns: {json.dumps(columns)}\nQuestion: {question}"
    problem = "no reply"
    for model in MODELS:
        try:
            result = client.interactions.create(
                model=model, input=prompt, store=False,
                response_format={"type": "text", "mime_type": "application/json", "schema": PLAN_SCHEMA})
            return json.loads(result.output_text), None
        except Exception as error:
            problem = type(error).__name__
    return None, problem

# ---------- Python checks the plan and does the calculation ----------

class CannotAnswer(Exception):
    pass

def real_column(df, name):
    simple = lambda t: re.sub(r"[\s_]+", "", str(t).lower())
    matches = [c for c in df.columns if str(c) == name] or [c for c in df.columns if simple(c) == simple(name)]
    if len(matches) != 1:
        raise CannotAnswer(f"I cannot find the column '{name}'.")
    return matches[0]

def as_numbers(df, col):
    """The column as numbers, or an error if it is mostly not numbers."""
    n = numbers(df, col)
    filled = df[col].notna().sum()
    if pd.api.types.is_bool_dtype(df[col]) or filled == 0 or n.notna().sum() < 0.9 * filled:
        raise CannotAnswer(f"'{col}' is not a number column.")
    return n

def to_number(text):
    try:
        return float(str(text).replace(",", ""))
    except ValueError:
        raise CannotAnswer(f"'{text}' is not a number.")

def one_filter(df, item):
    col, op, values = real_column(df, item["column"]), item["op"], item.get("values") or []
    blank = df[col].isna() | (df[col].astype(str).str.strip() == "")
    if op in ("empty", "not_empty"):
        return (blank if op == "empty" else ~blank), f"{col} is {'empty' if op == 'empty' else 'filled'}"
    if not values:
        raise CannotAnswer(f"No value was given for '{col}'.")
    label = f"{col} {op} {' or '.join(map(str, values))}"
    is_number = pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col])
    if op == "contains":
        text = df[col].astype(str).str.lower()
        mask = pd.Series(False, index=df.index)
        for value in values:
            mask |= text.str.contains(str(value).lower(), regex=False)
        return mask & ~blank, label
    if op in ("=", "!="):
        if is_number:
            mask = numbers(df, col).isin([to_number(v) for v in values])
        else:
            column = df[col].map(clean, na_action="ignore")
            existing = set(column.dropna())
            wanted = [clean(v) for v in values]
            for value in wanted:
                if value not in existing:
                    examples = ", ".join(sorted(existing)[:8])
                    raise CannotAnswer(f"'{value}' is not a value in '{col}'. Some values there: {examples}")
            mask = column.isin(wanted)
        return (mask if op == "=" else ~mask) & ~blank, label
    if op not in (">", ">=", "<", "<=") or len(values) != 1:
        raise CannotAnswer(f"I cannot apply that condition to '{col}'.")
    text = df[col].dropna().astype(str)
    if not is_number and len(text) and text.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
        column = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")
        limit = pd.to_datetime(str(values[0]), format="%Y-%m-%d", errors="coerce")
        if pd.isna(limit):
            raise CannotAnswer("Write the date as YYYY-MM-DD.")
    else:
        column, limit = as_numbers(df, col), to_number(values[0])
    compare = {">": column.gt, ">=": column.ge, "<": column.lt, "<=": column.le}[op]
    return compare(limit), label

def select(df, items, match="all"):
    masks = [one_filter(df, item) for item in items]
    keep = pd.Series(match != "any" or not masks, index=df.index)
    for mask, _ in masks:
        keep = (keep | mask) if match == "any" else (keep & mask)
    joiner = " or " if match == "any" else " and "
    return keep, joiner.join(label for _, label in masks)

def show_number(value):
    value = round(float(value), 2)
    return f"{value:.0f}" if value.is_integer() else f"{value:.2f}"

def run_plan(plan, df):
    if plan.get("action") == "cannot_answer":
        raise CannotAnswer(plan.get("reason") or "This cannot be answered from the table.")
    if plan.get("action") == "check":
        check = plan.get("check")
        if check not in CHECKS or plan.get("filters") or plan.get("among") or plan.get("group_by"):
            raise CannotAnswer("I cannot combine a data check with filters or groups.")
        if check == "duplicate_rows":
            return describe(df, check, None)
        cols = [real_column(df, plan["column"])] if plan.get("column") else columns_for(df, check)
        if check != "missing":
            cols = [c for c in cols if c in columns_for(df, check)]
        if not cols:
            raise CannotAnswer("No suitable column for that check.")
        return "\n".join(describe(df, check, c) for c in cols)

    kind = plan.get("calculation")
    names = {"count": "count rows", "percent": "percentage", "sum": "total", "mean": "average",
             "median": "median", "min": "lowest", "max": "highest", "distinct": "different values",
             "list": "show rows", "correlation": "relationship"}
    if kind not in names:
        raise CannotAnswer("I could not work out which calculation you need.")
    base, base_text = select(df, plan.get("among") or [])
    chosen, chosen_text = select(df, plan.get("filters") or [], plan.get("match", "all"))
    col = real_column(df, plan["column"]) if plan.get("column") else None
    group = real_column(df, plan["group_by"]) if plan.get("group_by") else None
    order, limit = plan.get("order") or "largest", max(int(plan.get("limit") or 0), 0)
    if group is not None and kind in ("list", "correlation", "distinct"):
        raise CannotAnswer("I cannot group that kind of question yet.")
    if kind in ("sum", "mean", "median", "min", "max", "distinct", "correlation") and col is None:
        raise CannotAnswer("I could not tell which column to use.")

    said = f"I understood: {names[kind]}" + (" of rows" if kind == "percent" else "")
    if col is not None and kind not in ("count", "percent"):
        said += f" of {col}"
    if kind == "correlation":
        said += f" and {plan.get('second_column')}"
    if chosen_text:
        said += f"; rows where {chosen_text}"
    if base_text:
        said += f"; within rows where {base_text}"
    if group is not None:
        said += f"; for each {group}"
    if limit:
        said += f"; {order} {limit}"
    rows_used = df[base & chosen]

    if kind == "percent":
        if not chosen_text:
            raise CannotAnswer("I could not tell what to take the percentage of.")
        def result(index):
            total = int(base.loc[index].sum())
            part = int((base & chosen).loc[index].sum())
            if total == 0:
                return None, ""
            return 100 * part / total, f"% ({part} of {total} rows)"
    elif kind == "count":
        def result(index):
            return int((base & chosen).loc[index].sum()), " rows"
    elif kind in ("sum", "mean", "median", "min", "max"):
        n = as_numbers(df, col)
        def result(index):
            values = n.loc[index][(base & chosen).loc[index]].dropna()
            if values.empty:
                return None, ""
            return getattr(values, kind)(), f" (from {len(values)} rows)"

    if kind in ("count", "percent", "sum", "mean", "median", "min", "max"):
        if group is None:
            value, note = result(df.index)
            if value is None:
                return f"{said}.\nNo matching data."
            if kind in ("sum", "mean", "median", "min", "max") and len(rows_used) > n[base & chosen].notna().sum():
                note += " Rows without a number were left out."
            return f"{said}.\nAnswer: {show_number(value)}{note}"
        keys = df[group].map(clean, na_action="ignore").fillna("(empty)")
        found = []
        for key, part in df.groupby(keys, sort=False):
            value, note = result(part.index)
            if value is not None and not (kind == "count" and value == 0):
                label = "(empty)" if key == "(empty)" else str(part[group].iloc[0]).strip()
                found.append((value, label, note))
        if not found:
            return f"{said}.\nNo matching data."
        found.sort(key=lambda x: x[0], reverse=order == "largest")
        wanted = limit or 15
        shown = found[:wanted]
        if limit:   # keep ties
            shown = [x for x in found if x in shown or x[0] == shown[-1][0]]
        lines = [f"- {label}: {show_number(value)}{note}" for value, label, note in shown[:15]]
        if len(found) > len(lines):
            lines.append(f"({len(lines)} of {len(found)} groups shown)")
        return "\n".join([said + "."] + lines)

    if rows_used.empty:
        return f"{said}.\nNo matching rows."
    if kind == "distinct":
        counts = rows_used[col].map(clean, na_action="ignore").value_counts()
        lines = [f"- {value}: {total}" for value, total in counts.head(15).items()]
        return "\n".join([f"{said}.", f"Answer: {len(counts)} different values (empty cells not counted)"] + lines)
    if kind == "correlation":
        other = real_column(df, plan.get("second_column") or "")
        pair = pd.DataFrame({"a": as_numbers(df, col), "b": as_numbers(df, other)}).loc[rows_used.index].dropna()
        if len(pair) < 3 or pair.a.nunique() < 2 or pair.b.nunique() < 2:
            raise CannotAnswer("There is not enough varied data to compare these columns.")
        r = pair.a.corr(pair.b)
        strength = "very weak or none" if abs(r) < 0.2 else "weak" if abs(r) < 0.4 else "moderate" if abs(r) < 0.7 else "strong"
        direction = "" if abs(r) < 0.2 else (", rising together" if r > 0 else ", one rises as the other falls")
        return (f"{said}.\nCorrelation: {r:.2f} ({strength}{direction}), from {len(pair)} rows. "
                "This measures a straight-line link only. It does not show cause.")
    if col is not None:
        try:
            key = as_numbers(df, col).loc[rows_used.index]
        except CannotAnswer:
            key = rows_used[col].astype(str).where(rows_used[col].notna())
        rows_used = rows_used.loc[key.sort_values(ascending=order == "smallest", na_position="last").index]
    wanted = min(limit or 5, 15)
    lines = [f"Row {i}: " + ", ".join(f"{c}={v}" for c, v in row.items()) for i, row in rows_used.head(wanted).iterrows()]
    return "\n".join([said + ".", f"Showing {len(lines)} of {len(rows_used)} rows:"] + lines)

## Choose how to answer

In [ ]:
CHECK_TALK = set("""any check are is there the a in of for ok values value rows row missing empty blank null duplicate
duplicates duplicated repeated negative unusual outlier outliers strange spelling spell invalid wrong valid correct
dates date id ids how many find show under over below above too young old 18 100 do we have column data""".split())

def simple_check(question, df):
    """True only if every word belongs to a plain data check, so nothing is ignored."""
    q = clean(re.sub(r"[?!.,]", " ", question.lower()))
    for c in sorted(df.columns, key=lambda c: -len(str(c))):
        for name in {clean(c), clean(str(c).replace("_", " "))}:
            q = re.sub(pattern_for(name), " ", q)
    return all(word in CHECK_TALK for word in q.split())

def answer(question, df, client=None):
    intent = simple_understand(question, df)
    action = intent["action"] if intent else None
    plain = clean(re.sub(r"[?!.]", " ", question.lower()))

    # 1. Fixed commands
    if action == "help":
        return HELP
    if action == "check_all":
        return check_all(df)
    if re.fullmatch(r"(show )?(summary|columns)|how many rows( are there)?", plain):
        return f"{len(df)} rows. Columns: {', '.join(map(str, df.columns))}."
    if re.fullmatch(r"(show )?row \d+", plain):
        n = int(plain.split()[-1])
        return df.iloc[n].to_string() if n < len(df) else f"Choose a row from 0 to {len(df) - 1}."

    # 2. Simple questions that Python fully understands
    local = data_question(question, df)
    refused = local is not None and local.startswith(("I am not sure", "'", "Name one", "Ask for one", "I found no numbers", "I cannot mix"))
    if local is not None and not refused:
        return local
    if action == "check" and simple_check(question, df):
        check, col = intent["check"], intent.get("column")
        if check == "duplicate_rows":
            return describe(df, check, None)
        allowed = columns_for(df, check)
        if col is not None and check != "missing" and col not in allowed:
            return f"That check does not suit the column '{col}'."
        cols = [col] if col is not None else allowed
        return "\n".join(describe(df, check, c) for c in cols) if cols else "No suitable column for that check."

    # 3. Everything else: the AI plans, Python calculates
    if client is not None:
        plan, problem = ai_plan(question, df, client)
        if plan is None:
            return f"The AI did not respond ({problem}). Try again in a minute."
        try:
            return run_plan(plan, df)
        except CannotAnswer as reason:
            return f"I could not answer: {reason}"
        except Exception as error:
            return f"I could not answer this safely ({type(error).__name__}). Try simpler words."
    if local is not None:
        return local + "\nThe AI is OFF. Turn it on for harder questions."
    return "I don't know. The AI is OFF, so only simple questions work. Type help for examples."

## 5. Tests (no API key needed)

In [ ]:
assert answer("how many ages are missing?", data) == "age: no missing values found."
assert "missing values: 1. Rows: 3." in answer("any missing city?", data)
assert answer("how many cities are missing?", data).startswith("city:")
assert "Rows: 5, 6." in answer("any duplicate rows?", data)
assert "Rows: 0, 2, 7." in answer("check spelling in city", data)
assert "Rows: 4." in answer("negative values in age", data)
assert "Rows: 3." in answer("are the dates ok?", data)
assert "Rows: 7." in answer("any unusual balance?", data)
assert answer("what is the average age?", data).startswith("Average age:")
assert "Count where city = berlin: 3 of 9" in answer("how many berlin?", data)
assert "I did not answer" in answer("how many are not in berlin?", data)
assert answer("tell me a joke", data).startswith("I don't know")
print("All chatbot tests passed")

## 6. Chat window
Optional AI: add `GEMINI_API_KEY` in Colab Secrets and allow notebook access. The bot also works without it for the example questions.

In [ ]:
client = None
try:
    from google import genai
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    print("AI understanding: ON")
except Exception as error:
    print(f"AI understanding: OFF ({type(error).__name__})")

In [ ]:
import io
from IPython.display import HTML, JSON, display
from google.colab import output

state = {"df": data}   # the table the bot is checking now

def bot_reply(question):
    try:
        text = answer(question, state["df"], client)
    except Exception as error:
        text = f"Sorry, something went wrong ({type(error).__name__})."
    return JSON({"text": text})

def bot_load(name, content):
    try:
        table = pd.read_csv(io.StringIO(content))
        if table.empty:
            raise ValueError("no rows")
        state["df"] = table
        text = f"Loaded {name} with {len(table)} rows. Type: check all"
    except Exception as error:
        text = f"Could not load the file ({type(error).__name__}). The old data is still loaded."
    return JSON({"text": text})

output.register_callback("bot.reply", bot_reply)
output.register_callback("bot.load", bot_load)

PAGE = """
<style>
#dc { position:relative; height:600px; font-family:Arial,sans-serif; }
#dc * { box-sizing:border-box; }
#dc-window { position:absolute; right:12px; bottom:86px; width:420px; max-width:calc(100% - 24px); height:500px;
  display:flex; flex-direction:column; background:#fff; color:#222; border:1px solid #ddd; border-radius:12px;
  box-shadow:0 6px 24px rgba(0,0,0,.2); overflow:hidden; }
#dc-window.hidden { display:none; }
#dc-head { display:flex; justify-content:space-between; align-items:center; padding:14px 16px;
  border-bottom:1px solid #ddd; font-weight:bold; font-size:16px; }
#dc-close { border:0; background:none; font-size:22px; cursor:pointer; color:#5b3fb5; }
#dc-messages { flex:1; overflow-y:auto; padding:14px; }
.dc-msg { max-width:88%; width:fit-content; margin:0 0 10px; padding:10px 13px; border-radius:10px;
  font-size:14px; line-height:1.45; white-space:pre-wrap; overflow-wrap:anywhere; }
.dc-bot { background:#ece8fa; color:#222; }
.dc-me { background:#5b3fb5; color:#fff; margin-left:auto; }
.dc-table { font-family:monospace; }
#dc-form { display:flex; gap:8px; margin:0 12px; padding:6px; border:1px solid #bbb; border-radius:8px; }
#dc-input { flex:1; min-width:0; border:0; outline:0; font-size:15px; background:#fff; color:#222; }
#dc-form button { border:0; border-radius:6px; cursor:pointer; padding:8px 12px; }
#dc-csv { background:#eee; color:#222; }
#dc-send { background:#5b3fb5; color:#fff; }
#dc-form button:disabled { opacity:.5; cursor:default; }
#dc-note { text-align:center; font-size:11px; color:#666; padding:8px; }
#dc-open { position:absolute; right:12px; bottom:12px; width:62px; height:62px; border-radius:50%; border:0;
  background:#5b3fb5; color:#fff; font-size:26px; cursor:pointer; box-shadow:0 4px 14px rgba(0,0,0,.3); }
</style>

<div id="dc">
  <div id="dc-window">
    <div id="dc-head"><span>&#128202; Ishu's DataCheck Bot</span><button id="dc-close" title="Close">&times;</button></div>
    <div id="dc-messages"></div>
    <form id="dc-form">
      <button type="button" id="dc-csv" title="Load a CSV file">CSV</button>
      <input id="dc-input" placeholder="Ask about your data" autocomplete="off">
      <button type="submit" id="dc-send" title="Send">&#10148;</button>
    </form>
    <input type="file" id="dc-file" accept=".csv" hidden>
    <div id="dc-note">AI sees your question and column names, not your rows. Check the answer.</div>
  </div>
  <button id="dc-open" title="Open or close chat">&#128172;</button>
</div>

<script>
(() => {
  const get = id => document.getElementById(id);
  const box = get("dc-messages"), input = get("dc-input"), file = get("dc-file");

  function say(text, who) {
    const div = document.createElement("div");
    div.className = "dc-msg " + who + (text.includes("   ") ? " dc-table" : "");
    div.textContent = text;
    box.appendChild(div);
    box.scrollTop = box.scrollHeight;
    return div;
  }

  function busy(on) { get("dc-send").disabled = on; get("dc-csv").disabled = on; }

  async function ask(name, args) {
    busy(true);
    const wait = say("Checking...", "dc-bot");
    try {
      const result = await google.colab.kernel.invokeFunction(name, args, {});
      wait.remove();
      say(result.data["application/json"].text, "dc-bot");
    } catch (error) {
      wait.remove();
      say("I am not connected. Click Runtime, then Restart session and run all.", "dc-bot");
    }
    busy(false);
    input.focus();
  }

  get("dc-form").addEventListener("submit", event => {
    event.preventDefault();
    const question = input.value.trim();
    if (!question || get("dc-send").disabled) return;
    input.value = "";
    say(question, "dc-me");
    ask("bot.reply", [question]);
  });

  get("dc-csv").onclick = () => file.click();
  file.onchange = () => {
    const chosen = file.files[0];
    file.value = "";
    if (!chosen) return;
    if (chosen.size > 1000000) { say("That file is too big. Use a CSV under 1 MB.", "dc-bot"); return; }
    say("Load " + chosen.name, "dc-me");
    chosen.text().then(content => ask("bot.load", [chosen.name, content]));
  };

  get("dc-close").onclick = () => get("dc-window").classList.add("hidden");
  get("dc-open").onclick = () => get("dc-window").classList.toggle("hidden");

  say("Hi! I am Ishu's data checker. Practice data is loaded.\\nType: check all\\nType help for more examples.\\n__AI__", "dc-bot");
})();
</script>
"""

status = "AI is ON." if client is not None else "AI is OFF: only simple questions work."
display(HTML(PAGE.replace("__AI__", status)))

## Limits
- It cannot answer everything. It does not explain why, predict, give advice, draw charts or join two files.
- The AI can misunderstand a question. Read the "I understood" line before trusting an answer.
- Without a Gemini key only simple questions and data checks work.
- The AI does not see values, so you may need the exact words used in your table.
- It does not remember earlier questions and does not change your data.
- Flagged values may be correct.
- The chat window works only inside Google Colab. It is not a public website.
- CSV files must be under 1 MB.
- AI answers can be slow when the free model is busy.